# Evaluación Comparativa de Dimensionalidad y Balanceo de Clases

Este notebook tiene como objetivo evaluar metodológicamente y comparar de forma cuantitativa el impacto de:
1. **Reducción de Dimensionalidad**: Comparar el uso del **Top 30 Híbrido** de variables de consenso, el **Top 10 Híbrido** y una reducción multivariante mediante **MCA (Análisis de Correspondencias Múltiples)** con 10 componentes.
2. **Técnicas de Balanceo de Clases**: Evaluar el desempeño bajo **Baseline (sin balanceo)**, **SMOTE** y **SMOTETomek** para mitigar el desbalance en los niveles de riesgo y tipos de violencia.

Los modelos evaluados son:
* **Random Forest (RF)**
* **XGBoost (XGB)**
* **LightGBM (LGBM)**
* **CatBoost (CatBoost)**

Se evalúan de manera robusta usando las semillas `42`, `52` y `62` de forma sistemática y un tamaño de muestra configurable (`SAMPLE_SIZE`) para evitar desbordamiento de memoria (OOM) en entornos de 16GB RAM.

In [ ]:
# Instalar dependencias faltantes de forma condicional y a prueba de fallos
import sys
libs_to_check = {
    'xgboost': 'xgboost',
    'lightgbm': 'lightgbm',
    'catboost': 'catboost',
    'prince': 'prince',
    'imblearn': 'imbalanced-learn',
    'sklearn': 'scikit-learn'
}

libs_to_install = []
for lib_import, lib_install in libs_to_check.items():
    try:
        __import__(lib_import)
    except ImportError:
        libs_to_install.append(lib_install)

if libs_to_install:
    print(f"Instalando dependencias necesarias: {libs_to_install}...")
    !{sys.executable} -m pip install {' '.join(libs_to_install)}
else:
    print("Todas las dependencias principales están instaladas en el kernel activo.")

In [ ]:
import gc
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import prince
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 1000)

## Carga de Datos y Parámetros Globales
Cargamos la base de modelado Parquet y los rankings de consenso híbrido de cada target.

In [ ]:
# Parámetros configurables
SAMPLE_SIZE = 50000
SEEDS = [42, 52, 62]
BASE_PATH = Path("/home/munasqa/MAESTRIA/opencode")

print("Cargando dataset base de modelado...")
df_base = pd.read_parquet(BASE_PATH / "base_modelado.parquet")

if "nivel_de_riesgo_victima" in df_base.columns and "nivel_riesgo_victima" not in df_base.columns:
    df_base = df_base.rename(columns={"nivel_de_riesgo_victima": "nivel_riesgo_victima"})

print(f"Dataset base cargado con dimensiones: {df_base.shape}")

## Módulos de Entrenamiento y Auditoría de Leakage
Definimos funciones genéricas para procesar los datos de manera limpia, evitando fuga de información (leakage) y entrenando clasificadores robustos.

In [ ]:
def get_clean_features(df, target, consensus_features):
    """
    Genera el subconjunto de características libre de leakage.
    """
    leakage_patterns = [
        'tipo_violencia', 'nivel_riesgo_victima', 
        'nivel_de_riesgo_victima', '_orig', '_lbl'
    ]
    
    clean_list = []
    for f in consensus_features:
        # Evitar leak directo
        if any(p in f for p in leakage_patterns) and f != target:
            continue
        clean_list.append(f)
        
    return clean_list

def get_models(seed):
    """
    Instancia los 4 clasificadores con la semilla especificada.
    """
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=100, max_depth=12, random_state=seed, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100, max_depth=6, random_state=seed, n_jobs=-1, 
            eval_metric='mlogloss', verbosity=0
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=100, max_depth=6, random_state=seed, n_jobs=-1, 
            verbosity=-1
        ),
        "CatBoost": CatBoostClassifier(
            iterations=100, depth=6, random_state=seed, verbose=0, thread_count=-1
        )
    }

def color_semaphore(val):
    """
    Asigna un emoji de semáforo según el F1-macro para visualización.
    """
    if val >= 0.75:
        return f"🟢 {val:.4f}"
    elif val >= 0.60:
        return f"🟡 {val:.4f}"
    else:
        return f"🔴 {val:.4f}"

## FASE 1: Reducción de Dimensionalidad - `tipo_violencia`
Evaluamos el desempeño del Top 30, Top 10 y MCA (10 componentes) bajo las tres semillas.

In [ ]:
target = "tipo_violencia"
print(f"=== FASE 1: Entrenando Target '{target}' ===")

# 1. Cargar features de consenso híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)
features_top10 = features_top30[:10]

# 2. Muestreo estratificado
df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

results = []

# 3. Iterar por semillas
for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Preprocesar MCA sobre el Top 30
    # MCA requiere que las variables de entrada se traten como string (categóricas)
    X_train_mca_str = X_train_b.astype(str)
    X_test_mca_str = X_test_b.astype(str)
    
    mca = prince.MCA(n_components=10, n_iter=3, random_state=seed, check_input=True)
    X_train_mca = mca.fit_transform(X_train_mca_str)
    X_test_mca = mca.transform(X_test_mca_str)
    
    # Clasificadores
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # Configuración A: Top 30
        clf.fit(X_train_b, y_train)
        preds_top30 = clf.predict(X_test_b)
        f1_top30 = f1_score(y_test, preds_top30, average='macro')
        
        # Configuración B: Top 10
        clf.fit(X_train_b[features_top10], y_train)
        preds_top10 = clf.predict(X_test_b[features_top10])
        f1_top10 = f1_score(y_test, preds_top10, average='macro')
        
        # Configuración C: MCA (10 componentes)
        clf.fit(X_train_mca, y_train)
        preds_mca = clf.predict(X_test_mca)
        f1_mca = f1_score(y_test, preds_mca, average='macro')
        
        results.append({
            "Semilla": seed,
            "Modelo": model_name,
            "F1_Top30": f1_top30,
            "F1_Top10": f1_top10,
            "F1_MCA10": f1_mca
        })
        
    del mca
    gc.collect()

# 4. Generar Reporte de Consolidación
df_res = pd.DataFrame(results)
df_grouped = df_res.groupby("Modelo")[["F1_Top30", "F1_Top10", "F1_MCA10"]].mean().reset_index()

# Añadir la semaforización a las métricas promedio
df_grouped_colored = df_grouped.copy()
for col in ["F1_Top30", "F1_Top10", "F1_MCA10"]:
    df_grouped_colored[col] = df_grouped_colored[col].apply(color_semaphore)

print(f"\n--- RESULTADOS FASE 1: {target.upper()} (Promedio 3 Semillas) ---")
from IPython.display import Markdown, display
display(Markdown(df_grouped_colored.to_markdown(index=False)))

# Guardar resultados de Fase 1 para persistencia
df_res.to_csv(BASE_PATH / f"benchmark_reduccion_semillas_{target}.csv", index=False)

## FASE 1: Reducción de Dimensionalidad - `nivel_riesgo_victima`
Evaluamos el desempeño del Top 30, Top 10 y MCA (10 componentes) bajo las tres semillas.

In [ ]:
target = "nivel_riesgo_victima"
print(f"=== FASE 1: Entrenando Target '{target}' ===")

# 1. Cargar features de consenso híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)
features_top10 = features_top30[:10]

# 2. Muestreo estratificado
df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

results = []

# 3. Iterar por semillas
for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Preprocesar MCA sobre el Top 30
    X_train_mca_str = X_train_b.astype(str)
    X_test_mca_str = X_test_b.astype(str)
    
    mca = prince.MCA(n_components=10, n_iter=3, random_state=seed, check_input=True)
    X_train_mca = mca.fit_transform(X_train_mca_str)
    X_test_mca = mca.transform(X_test_mca_str)
    
    # Clasificadores
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # Configuración A: Top 30
        clf.fit(X_train_b, y_train)
        preds_top30 = clf.predict(X_test_b)
        f1_top30 = f1_score(y_test, preds_top30, average='macro')
        
        # Configuración B: Top 10
        clf.fit(X_train_b[features_top10], y_train)
        preds_top10 = clf.predict(X_test_b[features_top10])
        f1_top10 = f1_score(y_test, preds_top10, average='macro')
        
        # Configuración C: MCA (10 componentes)
        clf.fit(X_train_mca, y_train)
        preds_mca = clf.predict(X_test_mca)
        f1_mca = f1_score(y_test, preds_mca, average='macro')
        
        results.append({
            "Semilla": seed,
            "Modelo": model_name,
            "F1_Top30": f1_top30,
            "F1_Top10": f1_top10,
            "F1_MCA10": f1_mca
        })
        
    del mca
    gc.collect()

# 4. Generar Reporte de Consolidación
df_res = pd.DataFrame(results)
df_grouped = df_res.groupby("Modelo")[["F1_Top30", "F1_Top10", "F1_MCA10"]].mean().reset_index()

df_grouped_colored = df_grouped.copy()
for col in ["F1_Top30", "F1_Top10", "F1_MCA10"]:
    df_grouped_colored[col] = df_grouped_colored[col].apply(color_semaphore)

print(f"\n--- RESULTADOS FASE 1: {target.upper()} (Promedio 3 Semillas) ---")
display(Markdown(df_grouped_colored.to_markdown(index=False)))

df_res.to_csv(BASE_PATH / f"benchmark_reduccion_semillas_{target}.csv", index=False)

## FASE 2: Comparativa de Técnicas de Balanceo de Categorías
Una vez comprobada la solidez y consistencia del **Top 30 Híbrido por Consenso** (con el mejor desempeño promedio en las tres semillas), lo utilizamos para evaluar los escenarios de balanceo:
1. **Baseline**: Sin balanceo.
2. **SMOTE** (Synthetic Minority Over-sampling Technique).
3. **SMOTETomek** (SMOTE + Tomek Links de remoción de ruido).

Se evalúan bajo los mismos clasificadores (RF, XGBoost, LightGBM, CatBoost) y semillas (`42`, `52`, `62`).

### FASE 2: Balanceo de Categorías - `tipo_violencia`

In [ ]:
target = "tipo_violencia"
print(f"=== FASE 2: Balanceo para Target '{target}' ===")

# Cargar variables del Top 30 Híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)

df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

bal_results = []

for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Instanciar balanceadores
    smote = SMOTE(random_state=seed)
    smotetomek = SMOTETomek(random_state=seed)
    
    # Aplicar balanceadores sobre el conjunto de train
    X_train_smote, y_train_smote = smote.fit_resample(X_train_b, y_train)
    X_train_tomek, y_train_tomek = smotetomek.fit_resample(X_train_b, y_train)
    
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # 1. Escenario Baseline
        clf.fit(X_train_b, y_train)
        preds_base = clf.predict(X_test_b)
        f1_base = f1_score(y_test, preds_base, average='macro')
        
        # 2. Escenario SMOTE
        clf.fit(X_train_smote, y_train_smote)
        preds_smote = clf.predict(X_test_b)
        f1_smote = f1_score(y_test, preds_smote, average='macro')
        
        # 3. Escenario SMOTETomek
        clf.fit(X_train_tomek, y_train_tomek)
        preds_tomek = clf.predict(X_test_b)
        f1_tomek = f1_score(y_test, preds_tomek, average='macro')
        
        bal_results.append({
            "Semilla": seed,
            "Modelo": model_name,
            "F1_Baseline": f1_base,
            "F1_SMOTE": f1_smote,
            "F1_SMOTETomek": f1_tomek
        })
        
    del smote, smotetomek
    gc.collect()

# Consolidar y Reportar
df_bal = pd.DataFrame(bal_results)
df_bal_grouped = df_bal.groupby("Modelo")[["F1_Baseline", "F1_SMOTE", "F1_SMOTETomek"]].mean().reset_index()

df_bal_colored = df_bal_grouped.copy()
for col in ["F1_Baseline", "F1_SMOTE", "F1_SMOTETomek"]:
    df_bal_colored[col] = df_bal_colored[col].apply(color_semaphore)

print(f"\n--- RESULTADOS FASE 2: BALANCEO DE {target.upper()} (Promedio 3 Semillas) ---")
display(Markdown(df_bal_colored.to_markdown(index=False)))

df_bal.to_csv(BASE_PATH / f"benchmark_balanceo_semillas_{target}.csv", index=False)

### FASE 2: Balanceo de Categorías - `nivel_riesgo_victima`

In [ ]:
target = "nivel_riesgo_victima"
print(f"=== FASE 2: Balanceo para Target '{target}' ===")

# Cargar variables del Top 30 Híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)

df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

bal_results = []

for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Instanciar balanceadores
    smote = SMOTE(random_state=seed)
    smotetomek = SMOTETomek(random_state=seed)
    
    # Aplicar balanceadores sobre el conjunto de train
    X_train_smote, y_train_smote = smote.fit_resample(X_train_b, y_train)
    X_train_tomek, y_train_tomek = smotetomek.fit_resample(X_train_b, y_train)
    
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # 1. Escenario Baseline
        clf.fit(X_train_b, y_train)
        preds_base = clf.predict(X_test_b)
        f1_base = f1_score(y_test, preds_base, average='macro')
        
        # 2. Escenario SMOTE
        clf.fit(X_train_smote, y_train_smote)
        preds_smote = clf.predict(X_test_b)
        f1_smote = f1_score(y_test, preds_smote, average='macro')
        
        # 3. Escenario SMOTETomek
        clf.fit(X_train_tomek, y_train_tomek)
        preds_tomek = clf.predict(X_test_b)
        f1_tomek = f1_score(y_test, preds_tomek, average='macro')
        
        bal_results.append({
            "Semilla": seed,
            "Modelo": model_name,
            "F1_Baseline": f1_base,
            "F1_SMOTE": f1_smote,
            "F1_SMOTETomek": f1_tomek
        })
        
    del smote, smotetomek
    gc.collect()

# Consolidar y Reportar
df_bal = pd.DataFrame(bal_results)
df_bal_grouped = df_bal.groupby("Modelo")[["F1_Baseline", "F1_SMOTE", "F1_SMOTETomek"]].mean().reset_index()

df_bal_colored = df_bal_grouped.copy()
for col in ["F1_Baseline", "F1_SMOTE", "F1_SMOTETomek"]:
    df_bal_colored[col] = df_bal_colored[col].apply(color_semaphore)

print(f"\n--- RESULTADOS FASE 2: BALANCEO DE {target.upper()} (Promedio 3 Semillas) ---")
display(Markdown(df_bal_colored.to_markdown(index=False)))

df_bal.to_csv(BASE_PATH / f"benchmark_balanceo_semillas_{target}.csv", index=False)